In [1]:
import pandas as pd
import json
import glob
import os
import re

from functools import reduce

In [2]:
# 1. Get a list of all your JSON files
file_pattern = './logs_isco_esco/temporary_results_*.json' 
files = glob.glob(file_pattern)

# Step 1: Group all DataFrames by their model/prompt label
grouped_data = {}

for file in files:
    if os.path.getsize(file) == 0:
        continue

    try:
        with open(file, 'r') as f:
            data = json.load(f)
        
        temp_df = pd.DataFrame(data)
        
        # Extract label (e.g., 'qwen unstructured')
        basename = os.path.basename(file)
        name_match = re.search(r'results_(.*?)_\d', basename)
        label = name_match.group(1).rstrip('_').replace('_', ' ') if name_match else "unknown"

        if label not in grouped_data:
            grouped_data[label] = []
        
        grouped_data[label].append(temp_df)

    except Exception as e:
        print(f"Error reading {file}: {e}")

# Step 2: For each label, "squash" multiple files into one high-density DF
final_model_dfs = []

for label, dfs in grouped_data.items():
    # Stack all files for this model vertically
    combined = pd.concat(dfs, ignore_index=True)
    
    # Sort so that if there are duplicates, we have a consistent pick 
    # (Optional: sort by a timestamp if you want the newest values to take priority)
    # combined = combined.sort_values('some_timestamp_column', ascending=False)

    # The "Squash": Group by ID and take the first non-null value for ISCO and ESCO
    # This fills gaps where one file had IDs 0-500 and another had 500-1000
    squashed = combined.groupby('id', as_index=False).first()

    # Rename to your specific format
    rename_map = {
        'ISCO': f'ISCO {label}',
        'ESCO': f'ESCO {label}'
    }
    squashed = squashed.rename(columns=rename_map)
    
    # Keep only the ID and the new model-specific columns
    cols_to_keep = ['id', f'ISCO {label}', f'ESCO {label}']
    squashed = squashed[squashed.columns.intersection(cols_to_keep)]
    
    final_model_dfs.append(squashed)

# Step 3: Horizontal merge of the distinct models
if final_model_dfs:
    main_df = final_model_dfs[0]
    for next_df in final_model_dfs[1:]:
        main_df = pd.merge(main_df, next_df, on='id', how='outer')
    
    main_df = main_df.sort_values('id').reset_index(drop=True)
    
    print(f"Final Shape: {main_df.shape}")
    print("\nSample of merged columns:")
else:
    print("No data processed.")

Final Shape: (10580, 13)

Sample of merged columns:


In [3]:
main_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10580 entries, 0 to 10579
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   id                          10580 non-null  int64 
 1   ISCO llama semi-structured  2999 non-null   object
 2   ESCO llama semi-structured  2999 non-null   object
 3   ISCO llama structured       2999 non-null   object
 4   ESCO llama structured       2999 non-null   object
 5   ISCO llama unstructured     530 non-null    object
 6   ESCO llama unstructured     530 non-null    object
 7   ISCO qwen semi-structured   10580 non-null  object
 8   ESCO qwen semi-structured   10580 non-null  object
 9   ISCO qwen structured        10580 non-null  object
 10  ESCO qwen structured        10580 non-null  object
 11  ISCO qwen unstructured      4355 non-null   object
 12  ESCO qwen unstructured      4355 non-null   object
dtypes: int64(1), object(12)
memory usage: 1.0+ MB


In [4]:
main_df.to_excel("../../outputs/raw_outputs/ISCO_ESCO_triples.xlsx")

In [5]:
# 1. Identify all unique model/prompt combinations from the columns
# We look for columns starting with 'ISCO ' or 'ESCO '
columns = main_df.columns
model_prompt_pairs = set()

for col in columns:
    if col.startswith('ISCO '):
        model_prompt_pairs.add(col.replace('ISCO ', ''))

todo = {}

for pair in model_prompt_pairs:
    # Split "qwen unstructured" into model_name and prompt_type
    # We use rsplit to handle models that might have spaces in their name
    parts = pair.rsplit(' ', 1)
    model_name = parts[0]
    prompt_type = parts[1] if len(parts) > 1 else "default"
    
    # 2. Find IDs where ISCO or ESCO is null for this pair
    isco_col = f'ISCO {pair}'
    esco_col = f'ESCO {pair}'
    
    # Logic: An ID needs work if EITHER ISCO or ESCO is missing
    missing_mask = main_df[isco_col].isna() | main_df[esco_col].isna()
    missing_ids = main_df.loc[missing_mask, 'id'].tolist()
    
    # 3. Build the nested dictionary
    if model_name not in todo:
        todo[model_name] = {}
    
    todo[model_name][prompt_type] = missing_ids

# 4. Save to todo.json
with open('todo_ISCO.json', 'w') as f:
    json.dump(todo, f)

print(f"todo.json created! Found {len(model_prompt_pairs)} model/prompt configurations.")

todo.json created! Found 6 model/prompt configurations.


In [6]:
# 1. Define the master lists based on your pipeline's needs
expected_models = ['qwen', 'gemma', 'llama'] 
# Added 'semi-structured' to match your Traceback error
expected_prompts = ['structured', 'unstructured', 'semi-structured'] 

# 2. Get the full list of all IDs (10,580 entries)
all_ids = main_df['id'].unique().tolist()

todo = {}

for model in expected_models:
    todo[model] = {}
    for prompt in expected_prompts:
        # Standardize the label used in columns: "model prompt"
        # We check both "model_prompt" and "model prompt" to be safe
        pair_label = f"{model} {prompt}"
        isco_col = f'ISCO {pair_label}'
        esco_col = f'ESCO {pair_label}'
        
        # Check if we have any existing data for this combination
        if isco_col in main_df.columns:
            # Find IDs where either ISCO or ESCO is NaN
            missing_mask = main_df[isco_col].isna() | main_df[f'ESCO {pair_label}'].isna()
            missing_ids = main_df.loc[missing_mask, 'id'].tolist()
            todo[model][prompt] = [int(i) for i in missing_ids] # Ensure they are Python ints
        else:
            # If the model/prompt is totally missing, ALL IDs are todos
            todo[model][prompt] = all_ids

# 3. Save to todo.json
with open('todo_ISCO.json', 'w') as f:
    json.dump(todo, f)

print(f"todo.json created. Included {len(expected_prompts)} prompt types for {len(expected_models)} models.")

todo.json created. Included 3 prompt types for 3 models.
